# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside parameters such as pI and normalized abundances across different samples.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load and parse Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access and print dataset summary
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Let's examine available record sets, fields and column `@id`s. In Croissant, record sets represent tables or data matrices, while fields/columns represent the variables in those tables. All references are by their `@id`.

In [ ]:
# List all available record sets
print("Record sets in this dataset (by @id and name):")
all_record_sets = metadata.record_sets
for rs in all_record_sets:
    print(f"  @id: {rs.id}\tname: {rs.name}")

if len(all_record_sets) == 0:
    print('No record sets are present in the high-level metadata; falling back to schema distribution file objects.')

In [ ]:
# Print fields (columns) for each record set
if len(all_record_sets):
    for record_set in all_record_sets:
        print(f"\nRecord set '@id': {record_set.id}  | name: {record_set.name if record_set.name else '-'}")
        print("Fields/columns (by @id and name):")
        for field in record_set.fields:
            # Not all fields may have names
            print(f"  @id: {field.id}\tname: {field.name if field.name else '-'}\ttype: {field.data_type if hasattr(field, 'data_type') else '-'}")
else:
    # If no record sets are provided in high-level schema, enumerate available DataFileObjects
    print('Falling back: Browsing distributions (data files):')
    for obj in getattr(metadata, 'distributions', []):
        print(obj)

## 3. Data Extraction

Load the data from a selected record set using its `@id`. If more than one record set exists, choose one or iterate over several. All fields and columns are referenced by their `@id` for future-proof and stable code.

In [ ]:
# For demonstration, we will load all available record sets into pandas DataFrames.
dataframes = {}

record_sets = [rs.id for rs in metadata.record_sets] if len(metadata.record_sets) > 0 else []

if len(record_sets) == 0:
    # If there are no explicit record sets, try to use the distribution/DataFileObject
    print('No record sets found. Attempting to load CSVs from distributions...')
    for d in getattr(metadata, 'distributions', []):
        print('\nDistribution:', d)
else:
    for record_set_id in record_sets:
        # All Croissant datasets refer to record sets by their @id, not friendly name.
        print(f"Loading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Rows: {len(df)} - showing first 5 rows:")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's explore the data! We'll select a numeric field by `@id`, filter by a threshold, normalize the field, and group by some categorical variable (when available).

In [ ]:
# If you want to select a specific record set (use @id shown in section 2), set it here
if len(dataframes):
    record_set_id = list(dataframes.keys())[0]  # Pick the first record set
    df = dataframes[record_set_id]

    # List fields in DataFrame for convenience
    print('Columns in the selected record set:')
    print(df.columns.tolist())

    # Let's try to find numeric fields by attempting to convert columns to floats
    numeric_fields = [c for c in df.columns if pd.to_numeric(df[c], errors='coerce').notnull().any()]
    if len(numeric_fields):
        numeric_field_id = numeric_fields[0]  # Use the first numeric field
        print(f"Using numeric field (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].dropna().astype(float).mean()  # Use mean as threshold
        # Filter
        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (total: {len(filtered_df)})")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[numeric_field_id + '_normalized'] = (
            pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - 
            pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
        ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
        print(f"Head after normalization for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Try to group by a column with few unique values (categorical)
        candidate_group_fields = [c for c in df.columns if df[c].nunique() < 20 and c != numeric_field_id]
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            print(f"Grouping by field (by @id): {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print("Mean by group:")
            display(grouped_df)
        else:
            print('No suitable grouping (categorical) field found in the columns.')
    else:
        print('No numeric fields found in this record set.')
else:
    print('No record set could be loaded for EDA.')

## 5. Visualization

Let's plot the distribution of one of the main numeric fields (using its `@id` for reproducible reference). This step requires `matplotlib` and `seaborn` libraries for visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes):
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Use the numeric field id from above if possible
    numeric_field_id = None
    numeric_fields = [c for c in df.columns if pd.to_numeric(df[c], errors='coerce').notnull().any()]
    if len(numeric_fields):
        numeric_field_id = numeric_fields[0]

    if numeric_field_id is not None:
        plt.figure(figsize=(8,5))
        sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
    else:
        print('No numeric field available for visualization.')
else:
    print('No data available to plot.')

## 6. Conclusion

- We loaded the FAIR² dataset Croissant schema and programmatically discovered record sets and fields via their `@id`.
- Data were loaded into pandas DataFrames using the `mlcroissant` library and analyzed using standard Python tools.
- Numeric fields were filtered, normalized, and grouped; distributions were visualized.

This notebook provides a reproducible template for exploring Mass Spectrometry data or any Croissant-compatible dataset. For deeper analysis, continue investigating additional fields, filtering criteria, or linking to protein accession databases.
